# Libraries

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# Functions

In [ ]:
def run_ts_cv(model, X, y, n_splits=5, model_name="model"):
    """
    Runs TimeSeriesSplit CV for any classifier.
    Computes:
        - Accuracy
        - ROC-AUC

    :Arguments:
        model: model to train
        X: feature matrix
        y: target vector
        n_splits: number of CV splits
        model_name: name of the model (for printing)

    :Return: Results dict.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)

    acc_scores = []
    auc_scores = []

    fold = 1
    for train_idx, val_idx in tscv.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        y_prob = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_prob)
        acc = accuracy_score(y_val, y_pred)

        acc_scores.append(acc)
        auc_scores.append(auc)

        fold += 1

    print(
        f"\n{model_name} | Mean ACC = {np.mean(acc_scores):.4f}, \
            Mean AUC = {np.nanmean(auc_scores):.4f}"
    )

# Preparing

In [ ]:
df = pd.read_parquet("../data/feature/stock_eod_features.parquet")

# Split dataset based on time - keep 10% test for final model evaluation
cutoff_date = df["date"].quantile(0.9)
print("Train/Test cutoff date:", cutoff_date.strftime("%Y-%m-%d"))

train_df = df[df["date"] < cutoff_date].copy()
test_df = df[df["date"] >= cutoff_date].copy()

# One-hot encode 'symbol' categorical feature
train_df["symbol"] = train_df["symbol"].astype("category")
test_df["symbol"] = test_df["symbol"].astype("category")

train_df = pd.get_dummies(train_df, columns=["symbol"], drop_first=False)
test_df = pd.get_dummies(test_df, columns=["symbol"], drop_first=False)

train_df = train_df.reindex(sorted(train_df.columns), axis=1)
test_df = test_df.reindex(sorted(test_df.columns), axis=1)

# Prepare feature matrix and target vector
TARGET = "target"
exclude = ["date", TARGET]

feature_columns = [column for column in train_df.columns if column not in exclude]
SELECTED_FEATURE = [
    "day_of_month",
    "return_lag_1",
    "return",
    "return_roll_mean_10",
    "return_roll_std_10",
    "return_roll_std_5",
    "return_roll_mean_5",
    "return_lag_10",
    "volume",
    "close_open",
    "high_low",
    "return_lag_2",
    "rsi_14",
    "return_lag_5",
    "adj_close",
    "macd_signal",
    "day_of_week",
    "month",
    "macd",
    "sma_10",
    "symbol_AMZN",
    "symbol_MSFT",
    "symbol_AAPL",
    "symbol_GOOGL",
    "symbol_NVDA",
    "symbol_TSLA",
    "symbol_META",
]

X_train_all = train_df[feature_columns].values
X_test_all = test_df[feature_columns].values

X_train_selected = train_df[SELECTED_FEATURE].values
X_test_selected = test_df[SELECTED_FEATURE].values

y_train = train_df[TARGET].values
y_test = test_df[TARGET].values

# Tuning model

## All features

In [ ]:
def rf_objective(trail):
    params = {
        'n_estimators': trail.suggest_int('n_estimators', 100, 1000),
        'max_depth': trail.suggest_int('max_depth', 3, 20),
        'min_samples_split': trail.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trail.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trail.suggest_categorical('max_features', ['sqrt', 'log2']),
        'bootstrap': trail.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestClassifier(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    fold_acc = []

    for train_idx, val_idx in tscv.split(X_train_all):
        X_tr, X_val = X_train_all[train_idx], X_train_all[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        fold_acc.append(acc)

    return float(np.mean(fold_acc))


study = optuna.create_study(direction='maximize')
study.optimize(rf_objective, n_trials=50, show_progress_bar=True)

print("Best trial:")
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

rf_best_params = trial.params

In [ ]:
def xgb_objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "eta": trial.suggest_float("eta", 0.01, 0.3),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda": trial.suggest_float("lambda", 0.0, 5.0),
        "alpha": trial.suggest_float("alpha", 0.0, 5.0),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "random_state": 42,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "n_jobs": -1
    }

    model = XGBClassifier(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    fold_acc = []

    for train_idx, val_idx in tscv.split(X_train_all):
        X_tr, X_val = X_train_all[train_idx], X_train_all[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        fold_acc.append(acc)

    return float(np.mean(fold_acc))


study = optuna.create_study(direction='maximize')
study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print("Best trial:") #
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

xgb_best_params = trial.params

In [ ]:
def cat_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000),
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "random_strength": trial.suggest_float("random_strength", 1.0, 20.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0),
        "random_seed": 42,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "task_type": "CPU",
        "verbose": False
    }

    model = CatBoostClassifier(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    fold_acc = []

    for train_idx, val_idx in tscv.split(X_train_all):
        X_tr, X_val = X_train_all[train_idx], X_train_all[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        fold_acc.append(acc)

    return float(np.mean(fold_acc))


study = optuna.create_study(direction='maximize')
study.optimize(cat_objective, n_trials=50, show_progress_bar=True)

print("Best trial:") #
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

cat_best_params = trial.params

In [ ]:
def lgbm_objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 16, 156),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),
        "random_state": 42,
        "objective": "binary",
        "metric": "auc",
        "n_jobs": -1,
        "boosting_type": "gbdt",
        "verbosity": -1
    }

    model = LGBMClassifier(**params)
    tscv = TimeSeriesSplit(n_splits=5)
    fold_acc = []

    for train_idx, val_idx in tscv.split(X_train_all):
        X_tr, X_val = X_train_all[train_idx], X_train_all[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        fold_acc.append(acc)

    return float(np.mean(fold_acc))


study = optuna.create_study(direction='maximize')
study.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

print("Best trial:") #
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

lgbm_best_params = trial.params

## Selected

In [ ]:
def rf_objective(trail):
    params = {
        'n_estimators': trail.suggest_int('n_estimators', 100, 1000),
        'max_depth': trail.suggest_int('max_depth', 3, 20),
        'min_samples_split': trail.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trail.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trail.suggest_categorical('max_features', ['sqrt', 'log2']),
        'bootstrap': trail.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestClassifier(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    fold_acc = []

    for train_idx, val_idx in tscv.split(X_train_selected):
        X_tr, X_val = X_train_selected[train_idx], X_train_selected[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        fold_acc.append(acc)

    return float(np.mean(fold_acc))


study = optuna.create_study(direction='maximize')
study.optimize(rf_objective, n_trials=50, show_progress_bar=True)

print("Best trial:")
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

rf_best_params = trial.params

In [ ]:
def xgb_objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "eta": trial.suggest_float("eta", 0.01, 0.3),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda": trial.suggest_float("lambda", 0.0, 5.0),
        "alpha": trial.suggest_float("alpha", 0.0, 5.0),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "random_state": 42,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "n_jobs": -1
    }

    model = XGBClassifier(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    fold_acc = []

    for train_idx, val_idx in tscv.split(X_train_selected):
        X_tr, X_val = X_train_selected[train_idx], X_train_selected[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        fold_acc.append(acc)

    return float(np.mean(fold_acc))


study = optuna.create_study(direction='maximize')
study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print("Best trial:") #
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

xgb_best_params = trial.params

In [ ]:
def cat_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000),
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "random_strength": trial.suggest_float("random_strength", 1.0, 20.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0),
        "random_seed": 42,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "task_type": "CPU",
        "verbose": False
    }

    model = CatBoostClassifier(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    fold_acc = []

    for train_idx, val_idx in tscv.split(X_train_selected):
        X_tr, X_val = X_train_selected[train_idx], X_train_selected[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        fold_acc.append(acc)

    return float(np.mean(fold_acc))


study = optuna.create_study(direction='maximize')
study.optimize(cat_objective, n_trials=50, show_progress_bar=True)

print("Best trial:") #
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

cat_best_params = trial.params

In [ ]:
def lgbm_objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 16, 156),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),
        "random_state": 42,
        "objective": "binary",
        "metric": "auc",
        "n_jobs": -1,
        "boosting_type": "gbdt",
        "verbosity": -1
    }

    model = LGBMClassifier(**params)
    tscv = TimeSeriesSplit(n_splits=5)
    fold_acc = []

    for train_idx, val_idx in tscv.split(X_train_selected):
        X_tr, X_val = X_train_selected[train_idx], X_train_selected[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        fold_acc.append(acc)

    return float(np.mean(fold_acc))


study = optuna.create_study(direction='maximize')
study.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

print("Best trial:") #
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

lgbm_best_params = trial.params

# Final model

In [ ]:
# Best params for Catboost
best_params = {
    "iterations": 539,
    "depth": 9,
    "learning_rate": 0.14667195461639018,
    "l2_leaf_reg": 5.0152300454458905,
    "random_strength": 12.61739742270242,
    "bagging_temperature": 1.7613540779077792,
}

cat_final = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
    random_state=42,
    **best_params
).fit(X_train_selected, y_train)

# predict on test
y_pred = cat_final.predict(X_test_selected)
y_proba = cat_final.predict_proba(X_test_selected)[:,1]

# metrics
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba) if y_proba is not None else float("nan")
print("Final CatBoost test accuracy:", round(acc,4))
print("Final CatBoost test ROC-AUC:", round(auc,4))

In [ ]:
# Generate classification report
report = classification_report(
    y_test, y_pred, target_names=["Down (0)", "Up (1)"], output_dict=True
)
report_df = pd.DataFrame(report).transpose()

# Plot classification report as heatmap
fig, axes = plt.subplots(1, figsize=(14, 10))

# 1. Classification report heatmap
sns.heatmap(
    report_df.iloc[:-3, :-1],  # Exclude 'support' and summary rows
    annot=True,
    fmt=".2f",
    cmap="Blues",
    ax=axes,
    cbar=True,
)
axes.set_title("Classification Report", fontsize=14)
axes.set_xlabel("Metrics")
axes.set_ylabel("Classes")

plt.tight_layout()
plt.show()